In [1]:
%load_ext autoreload
%autoreload 2
import requests
from dataclasses import dataclass
from ratingcalc import binsearch, calcperf

In [2]:
ratinglist = {}
ratinglist_json = requests.get("https://codeforces.com/api/user.ratedList").json()
for user in ratinglist_json["result"]:
    ratinglist[user["handle"]] = user["rating"]

print(f'Total rated users: {len(ratinglist)}')
print(list(ratinglist.items())[:10])

Total rated users: 130617
[('Benq', 3792), ('jiangly', 3728), ('EvenImage', 3682), ('zhoukangyang', 3670), ('Kevin114514', 3625), ('Geothermal', 3569), ('maroonrk', 3555), ('strapple', 3515), ('tourist', 3439), ('dXqwq', 3436)]


In [24]:
@dataclass(slots=True)
class ProblemResult:
    success: bool
    wa: int
    seconds: int


@dataclass(slots=True)
class Participant:
    handle: str
    d2_points: int
    d2_placing: int
    problems: list[ProblemResult]


@dataclass(slots=True)
class ContestInfo:
    duration: int
    d2_scores: list
    d1_scores: list
    d2_contestant_ratings: list
    d1_contestant_ratings: list
    d2_points_to_rank: list
    d1_points_to_rank: list
    contestant_results: list[Participant]


def scrape_div2(d2id: int) -> ContestInfo:
    d1id = d2id - 1
    d2_json = requests.get(f'https://codeforces.com/api/contest.standings?contestId={d2id}').json()
    d1_json = requests.get(f'https://codeforces.com/api/contest.standings?contestId={d1id}').json()
    dur = d2_json['result']['contest']['durationSeconds']
    d2_scores = [entry['points'] for entry in d2_json['result']['problems']]
    d1_scores = [entry['points'] for entry in d1_json['result']['problems']]
    d2_contestant_ratings = []
    d2_points_to_rank = []
    contestant_results = []

    for entry in d2_json['result']['rows']:
        if len(entry['party']['members']) > 1:
            continue
        handle = entry['party']['members'][0]['handle']
        rating = ratinglist.get(handle, 1400)  # the default internal rating for calculation is 1400
        r = []
        for problem in entry['problemResults']:
            r.append(ProblemResult(
                success=problem['points'] > 0,
                wa=problem['rejectedAttemptCount'],
                seconds=problem.get('bestSubmissionTimeSeconds', 999999)
            ))
        assert len(r) >= 4, 'sanity check for number of problems in this contest >= 4 failed'
        d2_contestant_ratings.append(rating)
        d2_points_to_rank.append(entry['points'])
        if max(r[0].seconds, r[1].seconds) > min(x.seconds for x in r[2:]):
            print(f'warn: contestant {handle} has invalid problem times (solved A/B before C/...), skipping them')
            continue
        contestant_results.append(Participant(handle, entry['points'], entry['rank']-1, r))

    d1_contestant_ratings = []
    d1_points_to_rank = []
    for entry in d1_json['result']['rows']:
        if len(entry['party']['members']) > 1:
            continue
        handle = entry['party']['members'][0]['handle']
        rating = ratinglist.get(handle, 1400)
        d1_contestant_ratings.append(rating)
        d1_points_to_rank.append(entry['points'])

    return ContestInfo(
        duration=dur,
        d2_scores=d2_scores,
        d1_scores=d1_scores,
        d2_contestant_ratings=d2_contestant_ratings,
        d1_contestant_ratings=d1_contestant_ratings,
        d2_points_to_rank=d2_points_to_rank,
        d1_points_to_rank=d1_points_to_rank,
        contestant_results=contestant_results
    )


def div2todiv1(c: ContestInfo, res: list[ProblemResult]) -> int:
    """Convert div 2 points to estimated div 1 points"""
    offset = max(res[0].seconds, res[1].seconds)
    d1_points = 0
    for i in range(2, len(res)):
        if res[i].success:
            new_seconds = res[i].seconds - offset
            new_minutes = new_seconds // 60
            original_points = c.d1_scores[i-2]
            contest_minutes = c.duration // 60
            points = original_points - int( 120 * original_points * new_minutes / (250 * contest_minutes) ) - 50 * res[i].wa
            points = max(points, original_points*0.3)
            print(f'  i={i}, {new_minutes:.0f}m, -{res[i].wa:.0f} ---> {points:.0f}/{original_points:.0f}')
            d1_points += points
        else:
            print(f'  i={i}, not successful')
    return d1_points


def analyze(c: ContestInfo) -> list[tuple[float, float]]:
    """Analyze the contest and return a list of (div 2 perf, estimated div 1 perf)."""
    d2_rating = calcperf(c.d2_contestant_ratings)
    d1_rating = calcperf(c.d1_contestant_ratings)
    ret = []

    for i, res in list(enumerate(c.contestant_results)):
        d2_points = res.d2_points
        d2_perf = binsearch(d2_rating, res.d2_placing)
        d1_points = div2todiv1(c, res.problems)
        # D1 place is index i s.t. us >= d1_scores[i]. So this kind of rounds up a tiny bit.
        l = 0
        r = len(c.d1_points_to_rank)
        c.d1_points_to_rank.append(0)  # sentinel value
        while l < r:
            mid = l + (r-l)//2
            if d1_points < c.d1_points_to_rank[mid]:
                l = mid + 1
            else:
                r = mid
        d1_place = l
        d1_perf = binsearch(d1_rating, d1_place)
        if i <= 2000:
            print(f'div 2 points: {d2_points:.0f} <{d2_perf}> [#{res.d2_placing}]  /  div 1 points: {d1_points:.0f} <{d1_perf}>')
        ret.append((d2_perf, d1_perf))
    return ret



In [25]:
c = scrape_div2(2240)
data = analyze(c)


warn: contestant _AirQwQ_ has invalid problem times (solved A/B before C/...), skipping them
warn: contestant gopal.thecoder has invalid problem times (solved A/B before C/...), skipping them
warn: contestant iamburger has invalid problem times (solved A/B before C/...), skipping them
warn: contestant Gamblr has invalid problem times (solved A/B before C/...), skipping them
warn: contestant wyh_ghm has invalid problem times (solved A/B before C/...), skipping them
warn: contestant Bubbleawa has invalid problem times (solved A/B before C/...), skipping them
warn: contestant 137QWQ has invalid problem times (solved A/B before C/...), skipping them
warn: contestant Sumith_chandra has invalid problem times (solved A/B before C/...), skipping them
warn: contestant anishkumar04 has invalid problem times (solved A/B before C/...), skipping them
warn: contestant rajvijay1504 has invalid problem times (solved A/B before C/...), skipping them
warn: contestant AZ_SENCS has invalid problem times (

In [20]:
print(c.d2_contestant_ratings)
print(len(c.d2_contestant_ratings))

[2177, 1977, 2122, 1965, 1648, 1660, 2044, 2010, 2062, 1973, 1957, 2027, 1959, 1676, 1973, 1944, 1967, 1902, 1998, 1704, 1692, 1610, 1518, 1833, 1886, 1738, 1978, 1931, 1953, 1985, 1713, 1971, 1988, 1949, 1939, 1994, 1938, 1944, 1826, 1743, 1946, 1932, 1743, 1960, 1984, 1752, 1938, 1940, 1973, 1717, 1761, 1935, 1958, 1923, 1898, 1953, 1698, 1635, 1935, 1536, 1389, 1536, 1895, 1968, 1886, 1937, 1674, 1951, 1562, 1911, 1843, 1843, 1788, 1895, 1619, 1880, 1943, 1478, 1550, 1249, 1598, 1776, 1788, 1761, 1822, 1772, 1821, 1725, 1695, 1623, 1836, 1679, 1632, 1527, 1498, 1582, 1895, 1564, 1887, 1545, 1583, 1762, 1603, 1472, 1827, 1598, 1880, 1696, 1911, 1657, 1650, 1546, 1865, 1565, 1927, 1772, 1489, 1771, 1593, 1936, 1852, 1592, 1934, 1627, 1907, 1713, 1885, 1938, 1733, 1891, 1706, 1552, 1493, 1594, 1794, 1554, 1717, 1929, 1551, 1628, 1883, 1809, 1502, 1801, 1524, 1829, 1619, 1543, 1620, 1718, 1626, 1615, 1880, 1825, 1485, 1581, 1840, 1885, 1573, 1769, 1360, 1568, 1435, 1429, 1528, 1514, 145

In [21]:
# if i was 2200, how many people would i lose to

loss_counter = 0.0
for x in c.d2_contestant_ratings:
    prob = 1.0 / (1.0 + pow(10.0, (2200-x)/400.0))
    loss_counter += prob

print(loss_counter)

116.50488355241708
